In [12]:
# -------------------------------------------------
# McEliece с кодом Гоппы (вместо Хэмминга)
# -------------------------------------------------
from sage.misc.prandom import shuffle, randint
import random  # для random.sample в error vector

# --- 1. Генерация ключей ---
def random_permutation_matrix(n, F):
    perm = list(range(n))
    shuffle(perm)
    P = matrix(F, n, n, lambda i, j: 1 if j == perm[i] else 0)
    return P, perm

F2 = GF(2)

# === ЗАМЕНА: Hamming → Goppa ===
# Параметры кода Гоппы
m = 4                      # степень расширения: F_{2^m}
Fq = GF(2^m)              # поле F_{16}
R.<x> = PolynomialRing(Fq)

# Находим неприводимый многочлен степени t (для t=2)
g = None
for a in Fq:
    for b in Fq:
        cand = x^2 + a*x + b
        if cand.is_irreducible():
            g = cand
            break
    if g:
        break

if g is None:
    raise RuntimeError("Не найден неприводимый многочлен")

# Локаторы: все α ∈ Fq, где g(α) ≠ 0
L = [alpha for alpha in Fq if g(alpha) != 0]
C = codes.GoppaCode(g, L)

n = C.length()
k = C.dimension()
d = C.minimum_distance()
t = (d - 1) // 2

G = C.generator_matrix()   # k x n, над GF(2)
print(f"Код Гоппы: [{n}, {k}, d≥{2*t+1}] → t = {t}")
print("Порождающая матрица G (первые 3 строки):")
print(G[:3])  # не печатаем всю, если большая

# --- 2. Генерируем S (k x k, невырожденную) ---
S = random_matrix(F2, k, k)
while not S.is_invertible():
    S = random_matrix(F2, k, k)
print(f"\nМатрица S (k x k): {S.dimensions()}")
print("S.is_invertible() =", S.is_invertible())

# --- 3. Генерируем P (n x n) ---
P, perm = random_permutation_matrix(n, F2)
print("Матрица P:", P.dimensions())

# --- 4. Открытая матрица Ĝ = S * G * P ---
G_hat = S * G * P
print("Ĝ shape:", G_hat.dimensions())

print(f"\n✅ Открытый ключ: (Ĝ, t) с t = {t}")
print(f"✅ Закрытый ключ: (S, G, P)")


Код Гоппы: [16, 8, d≥5] → t = 2
Порождающая матрица G (первые 3 строки):
[1 0 0 0 0 0 0 0 0 1 0 1 1 0 0 1]
[0 1 0 0 0 0 0 0 1 0 1 0 1 0 1 1]
[0 0 1 0 0 0 0 0 1 1 1 1 0 0 0 1]

Матрица S (k x k): (8, 8)
S.is_invertible() = True
Матрица P: (16, 16)
Ĝ shape: (8, 16)

✅ Открытый ключ: (Ĝ, t) с t = 2
✅ Закрытый ключ: (S, G, P)


In [13]:
# -------------------------------------------------
# Шифрование сообщения
# -------------------------------------------------
msg = "attack 16:00"

# Переводим строку в биты
bits = ''.join(format(ord(c), '08b') for c in msg)
print("\nБиты:", bits[:64] + "..." if len(bits) > 64 else bits)

# Дополняем до кратности k
if len(bits) % k != 0:
    bits = bits.ljust(len(bits) + (k - len(bits) % k), '0')

blocks = [bits[i:i+k] for i in range(0, len(bits), k)]
vectors = [vector(F2, [int(b) for b in block]) for block in blocks]

print(f"\nРазбито на {len(vectors)} блок(-а/ов) по {k} бит:")
for i, v in enumerate(vectors[:3], 1):  # первые 3
    print(f"m_{i} =", v)
if len(vectors) > 3:
    print("...")

# --- Вектор ошибок (исправлен: используем random.sample) ---
def random_error_vector(n, t, F=F2):
    z = [F(0)] * n
    if t > 0:
        for i in random.sample(range(n), t):  # ✅ надёжно
            z[i] = F(1)
    return vector(F, z)

# Шифрование
c_arr = []
print("\nВеса векторов ошибок:")
for m in vectors:
    z = random_error_vector(n, t, F2)
    print(z.hamming_weight(), end=' ')  # должно быть t
    c_arr.append(m * G_hat + z)
print()



Биты: 0110000101110100011101000110000101100011011010110010000000110001...

Разбито на 12 блок(-а/ов) по 8 бит:
m_1 = (0, 1, 1, 0, 0, 0, 0, 1)
m_2 = (0, 1, 1, 1, 0, 1, 0, 0)
m_3 = (0, 1, 1, 1, 0, 1, 0, 0)
...

Веса векторов ошибок:
2 2 2 2 2 2 2 2 2 2 2 2 


In [14]:
# -------------------------------------------------
# Расшифрование
# -------------------------------------------------
P_inv = P.inverse()

# Шаг 2: c_i * P_inv
c_hat_arr = [c * P_inv for c in c_arr]

# Шаг 3: ДЕКОДИРОВАНИЕ (вот главная замена!)
D = C.decoder()

# Вместо C.decode_to_message() — используем декодер + извлечение u
m_dec = []
for i, c_hat in enumerate(c_hat_arr):
    # 3a. Декодируем до кодового слова
    c_decoded = D.decode_to_code(c_hat)
    # 3b. Находим u: u * G = c_decoded → u = координаты c_decoded в базисе строк G
    # Так как G — порождающая, строки — базис кода.
    # Используем: решаем линейную систему u*G = c_decoded
    # (G.T * u.T = c_decoded.T) → u = (G.T \ c_decoded.T).transpose()
    try:
        #u = (G.transpose() \ c_decoded).transpose()  # u — строка 1 x k
        u = G.transpose().solve_right(c_decoded)
    except:
        # fallback: перебор (для небольших k)
        for cand in VectorSpace(F2, k):
            if cand * G == c_decoded:
                u = cand
                break
    # 3c. m = u * S^{-1}
    m = u * S.inverse()
    m_dec.append(m)

print("\n✅ Дешифрование завершено. Первые 2 блока:")
for i in range(min(2, len(m_dec))):
    print(f"m_rec_{i+1} =", m_dec[i])

# -------------------------------------------------
# Восстановление строки
# -------------------------------------------------
def vectors_to_string(vec_list):
    bits = ''.join(''.join(str(int(b)) for b in v) for v in vec_list)
    chars = []
    for i in range(0, len(bits), 8):
        byte = bits[i:i+8].ljust(8, '0')
        chars.append(chr(int(byte, 2)))
    return ''.join(chars).rstrip('\x00')  # убираем padding

recovered = vectors_to_string(m_dec)
print("Исходное сообщение:", repr(msg))
print("Расшифровано:", repr(recovered))
print("Успех?", recovered == msg)


✅ Дешифрование завершено. Первые 2 блока:
m_rec_1 = (0, 1, 1, 0, 0, 0, 0, 1)
m_rec_2 = (0, 1, 1, 1, 0, 1, 0, 0)
Исходное сообщение: 'attack 16:00'
Расшифровано: 'attack 16:00'
Успех? True
